# Setting up the API

In [12]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

In [13]:
load_dotenv()

True

In [14]:
api_key = os.getenv("GEMINI_API_KEY")

In [15]:
client = genai.Client(api_key=api_key)

In [16]:
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

# OpenAI version to check models

```python
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

for model in client.models.list():
    print(model.id)
```

# Anthropic version to check models

```python
import anthropic

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

for model in client.models.list(limit=100):
    print(model.id, "-", model.display_name)
```

# Generating Text

In [27]:
def generate_text(prompt):
    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=15,
            temperature=0.7,
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    return response.text.strip()

In [28]:
prompt = "Hi There! My name is Vaidheesh."

In [29]:
generated_text = generate_text(prompt)
print(prompt, generated_text)

Hi There! My name is Vaidheesh. Hello Vaidheesh! It's great to


# OpenAI (pip install openai) version of Generating Text

```python
from openai import OpenAI
client = OpenAI(api_key=api_key)
def generate_text(prompt):
    response = client.completions.create(
        model="davinci-002",
        prompt=prompt,
        max_tokens=10,
        temperature=0.7)
    return response.choices[0].text.strip()
```

# Gemini (pip install google-genai) version of Generating Text

```python
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def generate_text(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=10,
            temperature=0.7,
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    return response.text.strip()
```

# Anthropic (pip install anthropic) version of Generating Text

```python
import anthropic

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def generate_text(prompt):
    response = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=10,
        temperature=0.7,
        messages=[{"role": "user", "content": prompt}])
    return response.content[0].text.strip()
```

# Customizing the text output

In [47]:
def generate_text(prompt, max_output_tokens, temperature):
    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=max_output_tokens,
            temperature=temperature))

    return response.text.strip()

In [48]:
prompt = "Once upon a time, "

In [53]:
generated_text = generate_text(prompt, 1000, 0)
print(prompt, generated_text)

Once upon a time,  ...in a valley hidden behind the folds of a velvet sky, there lived a young cartographer named Silas. 

Silas didn't draw maps of mountains, rivers, or kingdom borders. He drew maps of things that moved—the migratory patterns of cloud-whales, the wandering paths of forgotten dreams, and the exact coordinates where the wind whispered its secrets.

One chilly autumn evening, as the silver leaves of the Whisper-Willows began to hum, a sharp knock sounded at his door. 

When Silas opened it, no one was there. Instead, resting on his doorstep was a intricate origami bird folded from obsidian paper. It glowed with a faint, pulsing violet light. As Silas leaned down, the bird unspooled itself, flattening into a parchment letter, and leaving behind a tiny, tarnished brass key.

Written on the parchment in shimmering ink were just three words:

*


In [54]:
generated_text = generate_text(prompt, 1000, 1)
print(prompt, generated_text)

Once upon a time,  ...in a valley hidden by perpetual twilight, there lived a clockmaker who didn't fix clocks. Instead, he repaired broken memories.

Travelers from distant lands would visit his tiny, brass-lit shop, carrying pocket watches that ticked not with time, but with the rhythm of forgotten moments. Inside each watch lay a thread of shimmering light—a lost childhood laugh, a faded promise, or a lingering regret.

One stormy evening, just as the clockmaker was closing his shutters, the shop bell chimed. A stranger in a dark cloak stood in the doorway, drenched in rain, holding a watch made of solid, unreflective black glass.

"It won't open," the stranger whispered, their voice trembling. "And it’s growing heavier every day."

***

*Do you want to continue the story from here, or shall I keep going?*


- Loading data (documents, websites, text)
- Splitting text into chunks
- Creating embeddings
- Storing embeddings in a vector database
- Retrieving relevant context
- Passing that context to a GPT model
- Getting a meaningful answer

# Text Summarization 

* **System messages** - Instructions to the model. How the model should respond and behave
* **User messages** - User Input (block of text)
* **Assistant messages** - Output to the user input (eg: the keyword extracted from the block of text)

In [55]:
system_prompt = "You will be provided with a block of text, and your task is to extract a list of keywords from it."

sample_content = [
    ("A flying saucer seen by a guest house, a 7ft alien-like figure coming out of a hedge and a \"cigar-shaped\" UFO near a school yard.\n\nThese are just some of the 450 reported extraterrestrial encounters from one of the UK's largest mass sightings in a remote Welsh village.\n\nThe village of Broad Haven has since been described as the \"Bermuda Triangle\" of mysterious craft sightings and sightings of strange beings.\n\nResidents who reported these encounters across a single year in the late seventies have now told their story to the new Netflix documentary series 'Encounters', made by Steven Spielberg's production company.\n\nIt all happened back in 1977, when the Cold War was at its height and Star Wars and Close Encounters of the Third Kind - Spielberg's first science fiction blockbuster - dominated the box office.",
     "flying saucer, guest house, 7ft alien-like figure, hedge, cigar-shaped UFO, school yard, extraterrestrial encounters, UK, mass sightings, remote Welsh village, Broad Haven, Bermuda Triangle, mysterious craft sightings, strange beings, residents, single year, late seventies, Netflix documentary series, Steven Spielberg, production company, 1977, Cold War, Star Wars, Close Encounters of the Third Kind, science fiction blockbuster, box office."),
    ("Each April, in the village of Maeliya in northwest Sri Lanka, Pinchal Weldurelage Siriwardene gathers his community under the shade of a large banyan tree. The tree overlooks a human-made body of water called a wewa – meaning reservoir or \"tank\" in Sinhala. The wewa stretches out besides the village's rice paddies for 175-acres (708,200 sq m) and is filled with the rainwater of preceding months.    \n\nSiriwardene, the 76-year-old secretary of the village's agrarian committee, has a tightly-guarded ritual to perform. By boiling coconut milk on an open hearth beside the tank, he will seek blessings for a prosperous harvest from the deities residing in the tree. \"It's only after that we open the sluice gate to water the rice fields,\" he told me when I visited on a scorching mid-April afternoon.\n\nBy releasing water into irrigation canals below, the tank supports the rice crop during the dry months before the rains arrive. For nearly two millennia, lake-like water bodies such as this have helped generations of farmers cultivate their fields. An old Sinhala phrase, \"wewai dagabai gamai pansalai\", even reflects the technology's centrality to village life; meaning \"tank, pagoda, village and temple\".",
     "April, Maeliya, northwest Sri Lanka, Pinchal Weldurelage Siriwardene, banyan tree, wewa, reservoir, tank, Sinhala, rice paddies, 175-acres, 708,200 sq m, rainwater, agrarian committee, coconut milk, open hearth, blessings, prosperous harvest, deities, sluice gate, rice fields, irrigation canals, dry months, rains, lake-like water bodies, farmers, cultivate, Sinhala phrase, technology, village life, pagoda, temple.")
]

In [56]:
def summarize_text(prompt):
    contents = []
    for sample_text, sample_keywords in sample_content:
        contents.append({"role": "user", "parts": [{"text": sample_text}]})
        contents.append({"role": "assistant", "parts": [{"text": sample_keywords}]})
    contents.append({"role": "user", "parts": [{"text": prompt}]})

    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0.5,
            max_output_tokens=1000))

    return response.text.strip()

In [57]:
prompt = "Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male manta ray trailing a female in perfect sync – an effort to impress a potential mate, exactly as Whitman had described during her animated presentation the previous evening. Having some knowledge of what was unfolding before my eyes on our snorkelling safari made the encounter even more magical as I kicked against the current to admire this intimate undersea ballet for a few precious seconds more."
print(prompt)

Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male manta ray trailing a female in perfect sync – an effort to impress a potential mate, exactly as Whitman had described during her animated presentation the previous evening. Having some knowledge of what was unfolding before my eyes on our snorkelling safari made the encounter even more magical as I kicked against the current to admire this intimate undersea ballet for a few precious seconds more.


In [58]:
summarized_text = summarize_text(prompt)
print(summarized_text)

Master Reef Guide, Kirsty Whitman, snorkel mask, huge male manta ray, female, potential mate, presentation, snorkelling safari, encounter, current, intimate undersea ballet.


# OpenAI version of Text Summarization and keyword extraction

```python
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def extract_keywords(text):
    messages = [{"role": "system", "content": system_prompt}]
    for example_text, example_keywords in examples:
        messages.append({"role": "user", "content": example_text})
        messages.append({"role": "assistant", "content": example_keywords})
    messages.append({"role": "user", "content": text})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.5,
        max_tokens=256)
    return response.choices[0].message.content.strip()
```

# Gemini version of Text summarization and keyword extraction

```python
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def extract_keywords(text):
    contents = []
    for example_text, example_keywords in examples:
        contents.append({"role": "user", "parts": [{"text": example_text}]})
        contents.append({"role": "model", "parts": [{"text": example_keywords}]})
    contents.append({"role": "user", "parts": [{"text": text}]})

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0.5,
            max_output_tokens=1000))
    return response.text.strip()
```

# Anthropic version of Text summarization and keyword extraction

```python
import anthropic

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def extract_keywords(text):
    messages = []
    for example_text, example_keywords in examples:
        messages.append({"role": "user", "content": example_text})
        messages.append({"role": "assistant", "content": example_keywords})
    messages.append({"role": "user", "content": text})

    response = client.messages.create(
        model="claude-sonnet-5-5",
        system=system_prompt,
        messages=messages,
        temperature=0.5,
        max_tokens=256)
    return response.content[0].text.strip()
```

# Langchain

An open source framework that allows Large Language Models (LLM) to combine with external sources of computation and data

* Loading data (documents, websites, text)
* Splitting text into chunks
* Creating embeddings
* Storing embeddings in a vector database
* Retrieving relevant context
* Passing that context to a GPT model
* Getting a meaningful answer

In [63]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [64]:
url = "https://365datascience.com/upcoming-courses"

In [65]:
loader = WebBaseLoader(url)

In [66]:
raw_documents = loader.load()

In [67]:
text_splitter = RecursiveCharacterTextSplitter()

In [68]:
documents = text_splitter.split_documents(raw_documents)

In [69]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001",
                                          google_api_key=os.getenv("GEMINI_API_KEY"))

In [70]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash",
                             google_api_key=os.getenv("GEMINI_API_KEY"),
                             temperature=0)

In [71]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [72]:
retriever = vectorstore.as_retriever()

In [79]:
from langchain.tools import tool
from langchain.agents import create_agent

In [80]:
@tool
def search_courses(query: str) -> str:
    """ Search the course page for information relevant to the query. """
    results = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in results)

In [81]:
memory = InMemorySaver()

In [82]:
qa = create_agent(
    model=llm,
    tools=[search_courses],
    system_prompt="Answer questions using the search_courses tool. If the answer isn't in the results, say you don't know.",
    checkpointer=memory
)

In [83]:
config = {"configurable": {"thread_id": "chat-1"}}

In [84]:
query = "What is the next course to be uploaded on 365DataScience platform?"

In [85]:
result = qa.invoke({"messages": [{"role": "user", "content": query}]}, config)

In [87]:
print(result["messages"][-1].text)

I don't know. The available course information does not specify which course is scheduled to be uploaded next on the 365 Data Science platform.


# Alternate method

In [88]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

C:\Users\vaidheeshk\AppData\Local\Temp\ipykernel_1472\3353824168.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
C:\Users\vaidheeshk\AppData\Local\miniconda3\envs\ai_365\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate

In [92]:
qa = ConversationalRetrievalChain.from_llm(ChatGoogleGenerativeAI(model="gemini-3.5-flash",
                                                                  google_api_key=os.getenv("GEMINI_API_KEY"),
                                                                  temperature=0),
                                            vectorstore.as_retriever(),
                                            memory=memory)

In [93]:
query = "What is the next course to be uploaded on 365DataScience platform?"

In [94]:
result = qa.invoke({"question": query})

In [95]:
print(result["answer"])

Based on the provided context, there is no information about which course will be uploaded next on the platform.
